# CurrencySafe: data cleaning and preprocessing (Notebook 1 of 2)


## Step 1: Imports

In [3]:
import os, shutil, hashlib, collections  # os = files/folders; shutil = delete whole folders; hashlib = file fingerprints; collections = counting
import numpy as np  # numpy = numerical library, used in the preprocessing preview
from PIL import Image  # Pillow's Image = opens image files to check they are valid and to read their format

## Step 2: Dataset path (a COPY of the original download)

In [4]:
dataset_path = r"C:\dsu\SEMESTERS\FIFTH SEM\Machine Learning\THEORY\project\abc"  # CHANGE THIS to a fresh COPY of the original dataset; never point it at the folder used for training
data_dir = os.path.join(dataset_path, "data", "data")  # builds the path of the folder that holds the "fake" and "real" folders

## Step 3: Starting counts

In [5]:
def count_images():  # defines a helper that counts the images in each class
    return {cls: sum(len(files) for _, _, files in os.walk(os.path.join(data_dir, cls))) for cls in ["fake", "real"]}  # for each class, visits all its folders and counts the files
print("Before cleaning:", count_images())  # expected {'fake': 2508, 'real': 4937}, a total of 7,445

Before cleaning: {'fake': 2508, 'real': 4937}


## Step 4: Remove the Rs 2000 notes (withdrawn from circulation, not useful in practice)

In [6]:
for cls in ["fake", "real"]:  # loop over the two classes
    folder_2000 = os.path.join(data_dir, cls, "2000")  # path of this class's Rs 2000 folder
    if os.path.exists(folder_2000):  # only act if the folder is still there, so running the cell twice is safe
        shutil.rmtree(folder_2000)  # deletes the folder and every image inside it
        print("Deleted:", folder_2000)  # tells which folder was deleted
print("After removing Rs 2000:", count_images())  # expected {'fake': 2363, 'real': 4680}, a total of 7,043

Deleted: C:\dsu\SEMESTERS\FIFTH SEM\Machine Learning\THEORY\project\abc\data\data\fake\2000
Deleted: C:\dsu\SEMESTERS\FIFTH SEM\Machine Learning\THEORY\project\abc\data\data\real\2000
After removing Rs 2000: {'fake': 2363, 'real': 4680}


## Step 5: Corrupt-file scan

In [7]:
bad_files = []  # empty list that will collect the paths of unreadable images
for cls in ["fake", "real"]:  # loop 1: the two classes
    for denom in os.listdir(os.path.join(data_dir, cls)):  # loop 2: each denomination folder
        folder = os.path.join(data_dir, cls, denom)  # full path of this denomination folder
        for fname in os.listdir(folder):  # loop 3: each file in that folder
            path = os.path.join(folder, fname)  # full path of this one image
            try:  # try to open the image; if that fails, jump to "except" instead of crashing
                with Image.open(path) as img:  # opens the file; "with" closes it again afterwards
                    img.verify()  # checks the file is a valid, uncorrupted image
            except Exception:  # runs only if opening or verifying failed
                bad_files.append(path)  # records the unreadable file
print("Bad files found:", len(bad_files))  # expected 0

Bad files found: 0


## Step 6: Find duplicate images using file fingerprints (MD5 hashing)

In [8]:
hashes = {}  # dictionary: fingerprint (hash) -> list of files that have that fingerprint
for cls in ["fake", "real"]:  # loop 1: the two classes
    for denom in os.listdir(os.path.join(data_dir, cls)):  # loop 2: each denomination folder
        folder = os.path.join(data_dir, cls, denom)  # full path of this denomination folder
        for fname in os.listdir(folder):  # loop 3: each file in that folder
            path = os.path.join(folder, fname)  # full path of this one image
            with open(path, "rb") as f:  # opens the file in "read binary" mode to read its raw bytes
                h = hashlib.md5(f.read()).hexdigest()  # turns the file's bytes into a fingerprint string; identical files always give the same one
            hashes.setdefault(h, []).append(path)  # stores the path under its fingerprint (creates the list the first time)
dupes = {h: p for h, p in hashes.items() if len(p) > 1}  # keeps only fingerprints shared by 2 or more files = the duplicates
print("Duplicate groups:", len(dupes), "| Extra copies:", sum(len(p) - 1 for p in dupes.values()))  # expected Duplicate groups: 164 | Extra copies: 217

Duplicate groups: 164 | Extra copies: 217


## Step 7: Safety check, no duplicate group may mix fake and real

In [9]:
cross = {h: p for h, p in dupes.items() if len({os.path.basename(os.path.dirname(os.path.dirname(x))) for x in p}) > 1}  # keeps groups whose files sit in more than one class folder (a labeling conflict)
print("Groups mixing fake and real:", len(cross))  # expected 0, so it is safe to keep one copy of each image

Groups mixing fake and real: 0


## Step 8: Remove the extra copies (keep the first file of each group)

In [10]:
for paths in dupes.values():  # loop over each group of identical images
    for extra in paths[1:]:  # every file in the group except the first one
        os.remove(extra)  # deletes that single duplicate file
print("After removing duplicates:", count_images())  # expected {'fake': 2289, 'real': 4537}, a total of 6,826

After removing duplicates: {'fake': 2289, 'real': 4537}


## Step 9: Convert AVIF images to JPEG (TensorFlow cannot read AVIF)

In [11]:
all_paths = [os.path.join(data_dir, c, d, f) for c in ["fake", "real"] for d in os.listdir(os.path.join(data_dir, c)) for f in os.listdir(os.path.join(data_dir, c, d))]  # list of every image path in the dataset
print(collections.Counter(Image.open(p).format for p in all_paths))  # reads each file's real format; expected JPEG 6813, PNG 10, AVIF 2, WEBP 1

Counter({'JPEG': 6813, 'PNG': 10, 'AVIF': 2, 'WEBP': 1})


In [12]:
for p in [p for p in all_paths if p.lower().endswith(".avif")]:  # loop over the AVIF files only
    with Image.open(p) as im:  # opens the AVIF image; "with" closes the file afterwards, which Windows needs before deleting
        im.convert("RGB").save(os.path.splitext(p)[0] + ".jpg", "JPEG")  # converts to plain RGB colour and saves a JPEG with the same name
    os.remove(p)  # deletes the AVIF original, so only the JPEG copy remains

## Step 10: Final checks

In [13]:
final_paths = [os.path.join(data_dir, c, d, f) for c in ["fake", "real"] for d in os.listdir(os.path.join(data_dir, c)) for f in os.listdir(os.path.join(data_dir, c, d))]  # list of every image path after cleaning
print(count_images(), "| Total:", len(final_paths))  # expected {'fake': 2289, 'real': 4537} | Total: 6826
print(collections.Counter(Image.open(p).format for p in final_paths))  # expected JPEG 6815, PNG 10, WEBP 1 and no AVIF

{'fake': 2289, 'real': 4537} | Total: 6826
Counter({'JPEG': 6815, 'PNG': 10, 'WEBP': 1})


## Step 11: Preprocessing preview (the same steps TensorFlow performs during training)

In [14]:
sample = Image.open(next(os.path.join(r, f) for r, _, fs in os.walk(os.path.join(data_dir, "real")) for f in fs))  # opens the first real-note image that it finds
print("Original size (width, height):", sample.size)  # the image's original size; the dataset has different sizes
resized = sample.convert("RGB").resize((224, 224))  # step 1: force 3 colour channels (RGB) and resize to 224 x 224, the size MobileNetV2 expects
arr = np.asarray(resized, dtype="float32") / 255.0  # step 2: convert to numbers and normalize pixel values from 0-255 to 0-1
print(arr.shape, arr.min().round(3), arr.max().round(3))  # expected shape (224, 224, 3) and values between 0.0 and 1.0

Original size (width, height): (482, 280)
(224, 224, 3) 0.031 1.0


**Next: Notebook 2 (CurrencySafe_clean.ipynb)** starts from this cleaned data. It builds the stratified 70/15/15 split, class weights, augmentation (flip, rotation, zoom, contrast), the MobileNetV2 model, evaluation and the valid-note check.